<a href="https://colab.research.google.com/github/rishita-svg/google-colab/blob/main/day1_manuscriptsextract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch torchvision Pillow

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image

# 1. PUT YOUR EXACT FILE NAME HERE:
image_name = "road of engineers_2.jpg"

print("Loading image...")
image = Image.open(image_name).convert("RGB")

print("Downloading the AI model (this takes a minute)...")
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

print("Extracting text...")
pixel_values = processor(image, return_tensors="pt").pixel_values
generated_ids = model.generate(pixel_values)
extracted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n" + "="*30)
print("--- EXTRACTED TEXT ---")
print("="*30)
print(extracted_text)

In [ ]:
!pip install -q accelerate torchvision
!pip install -q git+https://github.com/huggingface/transformers

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image

# 1. Ensure your correct file name is here:
image_name = "road of engineers_2.jpg"

print("Loading image...")
image = Image.open(image_name).convert("RGB")

print("Loading Qwen2-VL-2B Model...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

# MEMORY OPTIMIZATION: Set explicit pixel limits to avoid CUDA Out-Of-Memory
# This prevents the image from generating too many visual tokens.
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    min_pixels=256*256,
    max_pixels=512*512
)

# Formatting the visual chat template
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Please transcribe all the readable handwritten or printed text found in this historical document image accurately. Maintain the layout lines if possible."}
        ]
    }
]

print("Processing image and text prompt with pixel limits...")
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = processor(text=[text], images=image, padding=True, return_tensors="pt")
inputs = inputs.to("cuda")

print("The VLM is reading your manuscript within safe memory boundaries...")
# Added pad_token_id to keep the generation clean
generated_ids = model.generate(**inputs, max_new_tokens=512, pad_token_id=processor.tokenizer.pad_token_id)
generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

print("\n" + "="*30)
print("--- VLM EXTRACTED TEXT ---")
print("="*30)
print(output_text)

In [ ]:
!pip install -q timm einops

In [ ]:
# 1. Install required library versions for Florence-2
!pip install -q transformers==4.41.2 timm einops pdf2image
!apt-get install -y poppler-utils

In [ ]:
import torch
import os
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports
from unittest.mock import patch

# 1. THE PATCH: Trick the library scanner into ignoring flash_attn
def fixed_get_imports(filename: str | os.PathLike) -> list[str]:
    if not str(filename).endswith("modeling_florence2.py"):
        return get_imports(filename)
    imports = get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

# 2. Ensure your correct file name is here:
image_name = "road of engineers_2.jpg"

print("Loading high-resolution image...")
image = Image.open(image_name).convert("RGB")

print("Loading Microsoft Florence-2-large (Bypassing flash_attn requirement)...")

# 3. Apply the patch silently while the model loads
with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):
    model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Florence-2-large",
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to("cuda")

processor = AutoProcessor.from_pretrained(
    "microsoft/Florence-2-large",
    trust_remote_code=True
)

prompt = "<OCR>"

print("Processing prompt and image...")
inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda")
inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

print("The VLM is transcribing the high-res document text...")
with torch.no_grad():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        do_sample=False,
        num_beams=3
    )

output_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n" + "="*30)
print("--- FLORENCE-2 EXTRACTED TEXT ---")
print("="*30)
print(output_text)

In [ ]:
!pip install -q transformers==4.41.2

In [ ]:
import torch
import os
import glob
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports
from unittest.mock import patch

# 1. Flash-Attention patch to ensure smooth loading
def fixed_get_imports(filename: str | os.PathLike) -> list[str]:
    if not str(filename).endswith("modeling_florence2.py"):
        return get_imports(filename)
    imports = get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

print("Initializing Florence-2-large...")
with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):
    model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Florence-2-large",
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to("cuda")

processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

# 2. Find all images in your current directory (.jpg, .jpeg, .png)
# Change path if your files are inside a specific folder (e.g., "/content/my_folder/*.jpg")
image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
image_files = []
for ext in image_extensions:
    image_files.extend(glob.glob(f"/content/{ext}"))

image_files = sorted(list(set(image_files)))
print(f"Found {len(image_files)} images to process.")

# 3. Open a text file to write the final results
output_report_path = "/content/transcription_results.txt"

with open(output_report_path, "w") as f:
    f.write("==================================================\n")
    f.write("         BATCH EXTRACTION REPORT                  \n")
    f.write("==================================================\n\n")

    for idx, img_path in enumerate(image_files, 1):
        filename = os.path.basename(img_path)
        print(f"[{idx}/{len(image_files)}] Processing {filename}...")

        try:
            # Load and process image
            image = Image.open(img_path).convert("RGB")
            inputs = processor(text="<OCR>", images=image, return_tensors="pt").to("cuda")
            inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

            with torch.no_grad():
                generated_ids = model.generate(
                    input_ids=inputs["input_ids"],
                    pixel_values=inputs["pixel_values"],
                    max_new_tokens=1024,
                    do_sample=False,
                    num_beams=3
                )

            output_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

            # Save result to file
            f.write(f"--- FILE: {filename} ---\n")
            f.write(output_text + "\n")
            f.write("-" * 50 + "\n\n")

        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            f.write(f"--- FILE: {filename} ---\n")
            f.write(f"ERROR DURING PROCESSING: {str(e)}\n")
            f.write("-" * 50 + "\n\n")

print(f"\nBatch processing complete! Your results are saved in: {output_report_path}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q /content/manuscripts.zip -d /content/manuscripts_folder

In [ ]:
# 1. Update the server's package list so it finds the newest version
!apt-get update

# 2. Install the PDF conversion tools
!apt-get install -y poppler-utils
!pip install -q pdf2image

In [ ]:
import torch
import os
import glob
from PIL import Image
from pdf2image import convert_from_path
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports
from unittest.mock import patch

# 1. Flash-Attention patch for Florence-2 environment handling
def fixed_get_imports(filename: str | os.PathLike) -> list[str]:
    if not str(filename).endswith("modeling_florence2.py"):
        return get_imports(filename)
    imports = get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

print("Initializing Florence-2-large pipeline...")
with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):
    model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Florence-2-large",
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to("cuda")
processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

# 2. Set your input directory path here!
# If you used Method 1 (Google Drive), use: "/content/drive/MyDrive/manuscripts"
# If you used Method 2 (Zip file), use: "/content/manuscripts_folder/manuscripts"
SOURCE_DIR = "/content/drive/MyDrive/manuscripts"
OUTPUT_REPORT = "/content/final_transcription_report.txt"

# Gather all files from the directory
all_files = glob.glob(os.path.join(SOURCE_DIR, "*"))
print(f"Total files found in directory: {len(all_files)}")

with open(OUTPUT_REPORT, "w") as report:
    report.write("==================================================\n")
    report.write("       MASTER MANUSCRIPT EXTRACTION REPORT        \n")
    report.write("==================================================\n\n")

    for idx, file_path in enumerate(all_files, 1):
        filename = os.path.basename(file_path)
        ext = filename.split('.')[-1].lower()

        # Skip hidden files or system files
        if filename.startswith('.'): continue

        print(f"[{idx}/{len(all_files)}] Processing: {filename}")
        report.write(f"=== START OF FILE: {filename} ===\n")

        try:
            # FRAMEWORK BRANCH A: If the file is a PDF
            if ext == 'pdf':
                print(f"-> Converting PDF pages to high-res images...")
                pages = convert_from_path(file_path, dpi=300)
                for page_num, page_img in enumerate(pages, 1):
                    inputs = processor(text="<OCR>", images=page_img, return_tensors="pt").to("cuda")
                    inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

                    with torch.no_grad():
                        generated_ids = model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=1024, num_beams=3)
                    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                    report.write(f"[Page {page_num}]\n{text}\n")

            # FRAMEWORK BRANCH B: If the file is a standard Image (PNG, JPG, JPEG)
            elif ext in ['jpg', 'jpeg', 'png', 'tiff', 'bmp']:
                image = Image.open(file_path).convert("RGB")
                inputs = processor(text="<OCR>", images=image, return_tensors="pt").to("cuda")
                inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

                with torch.no_grad():
                    generated_ids = model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=1024, num_beams=3)
                text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                report.write(f"{text}\n")

            else:
                report.write(f"Skipped: Unsupported file extension .{ext}\n")

        except Exception as e:
            print(f"-> Error processing {filename}: {str(e)}")
            report.write(f"ERROR ENCOUNTERED: {str(e)}\n")

        report.write("-" * 60 + "\n\n")

print(f"\nProcessing complete! All transcriptions compiled into: {OUTPUT_REPORT}")

In [ ]:
import torch
import os
import glob
from PIL import Image
from pdf2image import convert_from_path
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports
from unittest.mock import patch

# 1. Flash-Attention patch for Florence-2 environment handling
def fixed_get_imports(filename: str | os.PathLike) -> list[str]:
    if not str(filename).endswith("modeling_florence2.py"):
        return get_imports(filename)
    imports = get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

print("Initializing Florence-2-large pipeline...")
with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):
    model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Florence-2-large",
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to("cuda")
processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

# 2. Set your input and output directories
# (Make sure SOURCE_DIR points to wherever your files are currently located!)
SOURCE_DIR = "/content/drive/MyDrive/manuscripts"
OUTPUT_DIR = "/content/extracted_texts"

# Create the output directory if it doesn't exist yet
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Gather all files from the directory
all_files = glob.glob(os.path.join(SOURCE_DIR, "*"))
print(f"Total files found in directory: {len(all_files)}\n")

for idx, file_path in enumerate(all_files, 1):
    filename = os.path.basename(file_path)
    ext = filename.split('.')[-1].lower()

    # Skip hidden files or folders
    if filename.startswith('.') or os.path.isdir(file_path):
        continue

    print(f"[{idx}/{len(all_files)}] Processing: {filename}")

    # 3. Create a matching text file name (e.g., "map.jpg" -> "map.txt")
    base_name = os.path.splitext(filename)[0]
    output_file_path = os.path.join(OUTPUT_DIR, f"{base_name}.txt")

    try:
        # Open the new specific text file to write into
        with open(output_file_path, "w", encoding="utf-8") as out_file:

            # FRAMEWORK BRANCH A: If the file is a PDF
            if ext == 'pdf':
                print(f"  -> Converting PDF pages...")
                pages = convert_from_path(file_path, dpi=300)
                for page_num, page_img in enumerate(pages, 1):
                    inputs = processor(text="<OCR>", images=page_img, return_tensors="pt").to("cuda")
                    inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

                    with torch.no_grad():
                        generated_ids = model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=1024, num_beams=3)
                    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                    out_file.write(f"[Page {page_num}]\n{text}\n\n")

            # FRAMEWORK BRANCH B: If the file is a standard Image
            elif ext in ['jpg', 'jpeg', 'png', 'tiff', 'bmp']:
                image = Image.open(file_path).convert("RGB")
                inputs = processor(text="<OCR>", images=image, return_tensors="pt").to("cuda")
                inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

                with torch.no_grad():
                    generated_ids = model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=1024, num_beams=3)
                text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                out_file.write(f"{text}\n")

            else:
                print(f"  -> Skipped: Unsupported file extension .{ext}")

    except Exception as e:
        print(f"  -> Error processing {filename}: {str(e)}")
        # If it crashes on a specific file, write the error into that text file
        with open(output_file_path, "w", encoding="utf-8") as err_file:
            err_file.write(f"ERROR ENCOUNTERED DURING EXTRACTION: {str(e)}\n")

print(f"\nProcessing complete! All 11 text files are saved in: {OUTPUT_DIR}")

In [ ]:
!zip -r /content/all_transcriptions.zip /content/extracted_texts

In [ ]:
!python -m spacy download en_core_web_md

In [ ]:
import os
import glob
import re
import pandas as pd
import spacy

# Load the spaCy NLP model
print("Loading NLP engine...")
nlp = spacy.load("en_core_web_md")

INPUT_DIR = "/content/extracted_texts"
OUTPUT_CSV = "/content/historical_entities.csv"

def clean_historical_text(raw_text):
    """
    Stage 1: Cleans up obvious OCR spacing and formatting anomalies
    without distorting the unique vocabulary of historical text.
    """
    # Remove repetitive back-to-back punctuation anomalies (e.g., //, --, ...)
    cleaned = re.sub(r'[-–—/_]{2,}', ' ', raw_text)
    # Standardize whitespace and line breaks
    cleaned = re.sub(r'\s+', ' ', cleaned)
    return cleaned.strip()

def extract_historical_entities(text):
    """
    Stage 2: Identifies critical historical markers using NER.
    """
    doc = nlp(text)
    entities = []

    # Target specific entity categories relevant to historical documents
    target_labels = {"PERSON", "GPE", "LOC", "DATE", "ORG"}

    for ent in doc.ents:
        if ent.label_ in target_labels:
            entities.append({
                "Text": ent.text.strip(),
                "Category": ent.label_
            })
    return entities

# Pipeline Execution Loop
all_extracted_data = []
txt_files = glob.glob(os.path.join(INPUT_DIR, "*.txt"))

print(f"Beginning pipeline on {len(txt_files)} text files...")

for file_path in txt_files:
    filename = os.path.basename(file_path)
    print(f"Processing: {filename}")

    with open(file_path, "r", encoding="utf-8") as f:
        raw_content = f.read()

    # Run Stage 1
    clean_content = clean_historical_text(raw_content)

    # Run Stage 2
    found_entities = extract_historical_entities(clean_content)

    # Organize data by document origin
    for entity in found_entities:
        all_extracted_data.append({
            "Document": filename,
            "Cleaned Text Snippet": entity["Text"],
            "Entity Type": entity["Category"]
        })

# Step 3: Export results to a structured format
df = pd.DataFrame(all_extracted_data)
# Remove exact duplicates within the same document to keep the spreadsheet tidy
df = df.drop_duplicates()
df.to_csv(OUTPUT_CSV, index=False)

print(f"\nPipeline successfully completed! Report saved to: {OUTPUT_CSV}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the extracted data
csv_path = "/content/historical_entities.csv"
try:
    df = pd.read_csv(csv_path)
    print(f"Successfully loaded {len(df)} historical entities for analysis.\n")
except FileNotFoundError:
    print("Error: CSV not found. Make sure the extraction script finished running!")
    df = pd.DataFrame()

if not df.empty:
    # Set up the visual style
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(16, 6))

    # --- CHART 1: Top Historical Figures (PERSON) ---
    plt.subplot(1, 2, 1)
    people_df = df[df["Entity Type"] == "PERSON"]
    if not people_df.empty:
        top_people = people_df["Cleaned Text Snippet"].value_counts().head(10)
        sns.barplot(x=top_people.values, y=top_people.index, palette="viridis")
        plt.title("Top 10 Mentioned Historical Figures", fontsize=14, fontweight='bold')
        plt.xlabel("Number of Mentions")
        plt.ylabel("")
    else:
        plt.text(0.5, 0.5, "No PERSON entities found.", ha='center', va='center')

    # --- CHART 2: Top Locations (GPE / LOC) ---
    plt.subplot(1, 2, 2)
    locations_df = df[df["Entity Type"].isin(["GPE", "LOC"])]
    if not locations_df.empty:
        top_locations = locations_df["Cleaned Text Snippet"].value_counts().head(10)
        sns.barplot(x=top_locations.values, y=top_locations.index, palette="magma")
        plt.title("Top 10 Mentioned Geographic Locations", fontsize=14, fontweight='bold')
        plt.xlabel("Number of Mentions")
        plt.ylabel("")
    else:
        plt.text(0.5, 0.5, "No Location entities found.", ha='center', va='center')

    # Adjust layout and display
    plt.tight_layout()
    plt.savefig("/content/historical_analytics_dashboard.png", dpi=300)
    plt.show()

    print("Analytics complete! Dashboard saved as 'historical_analytics_dashboard.png'.")